# EXP-010: Fresh-seed generalization check for auto_grad and the val-gap diagnostic (R6)

**Status: engineering validation / proof of concept**, same caveats as all prior
notebooks this session.

**Purpose (R6 of the revision roadmap, Methodology/Devil's Advocate):** every
headline accuracy number so far (`auto_grad`'s ~92-100% diagnostic accuracy,
EXP-008's val-gap 100% true-positive / 30-40% false-positive rates) was computed
on the SAME 10 seeds (0-9) used throughout development -- there is a real risk
those seeds were implicitly selected-on (by iterating on the diagnostic design
until it worked well on the seeds being watched), which would make the reported
accuracy an overfit estimate, not a generalization estimate. This notebook
re-runs BOTH diagnostics, UNCHANGED, on 20 completely fresh seeds (100-119)
never used or looked at during any prior notebook in this session, and reports
whether the numbers hold.

**auto_grad**: uses the R3-corrected post-convergence calibration window
(steps 150-199), since that is now the best-known-correct version of this
diagnostic -- testing the superseded warmup-calibrated version on fresh seeds
would just re-confirm a design already known to have a false-alarm-rate problem.

**val-gap (EXP-008)**: calibration and verdict logic are UNCHANGED from
EXP-008 exactly as it stands after its R1 correction (still uses the
non-stationary warmup window for its OWN threshold/baseline-gap calibration --
that specific issue was not in R3's scope and is a known, separate caveat,
not fixed here).


In [1]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd

torch.set_num_threads(1)
print("torch", torch.__version__, "| numpy", np.__version__)


C:\Users\USER\anaconda3\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


torch 2.5.1+cpu | numpy 2.0.2


## 1. Shared toy task/disturbance definitions

In [2]:
def make_model(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))

MEAN0_BASE = np.array([-1.0, -1.0])
MEAN1_BASE = np.array([1.0, 1.0])
BASE_NOISE_SCALE = 0.5

def gen_batch(rng, drift_frac=0.0, flip_prob=0.0, batch_size=16):
    frac = float(np.clip(drift_frac, 0.0, 1.0))
    mean0 = (1 - frac) * MEAN0_BASE + frac * MEAN1_BASE
    mean1 = (1 - frac) * MEAN1_BASE + frac * MEAN0_BASE
    y_true = rng.integers(0, 2, size=batch_size)
    means = np.where(y_true[:, None] == 0, mean0, mean1)
    x = means + rng.normal(scale=BASE_NOISE_SCALE, size=(batch_size, 2))
    flip_mask = rng.random(batch_size) < flip_prob
    y_observed = np.where(flip_mask, 1 - y_true, y_true)
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y_observed, dtype=torch.float32)

def grad_norm(model):
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += float((p.grad.detach() ** 2).sum())
    return total ** 0.5

N_STEPS = 400
DISTURBANCE_ONSET = 200
RAMP_LEN = 40
GRAD_NOISE_MAX = 8.0
LABEL_FLIP_PROB = 0.35
TREND_WINDOW = 5
GRAD_RATIO_SENSITIVITY = 1.5
CALIB_START, CALIB_END = 150, 200   # R3-corrected, used for auto_grad only
WARMUP = 50                          # unchanged, used for EXP-008 val-gap only

def _ramp(step):
    if step < DISTURBANCE_ONSET:
        return 0.0
    return min(step - DISTURBANCE_ONSET, RAMP_LEN) / RAMP_LEN

def _instant(step):
    return 1.0 if step >= DISTURBANCE_ONSET else 0.0

def drift_forecastable(step):
    return (_ramp(step), 0.0, 0.0)

def drift_unforecastable(step):
    return (_instant(step), 0.0, 0.0)

def instability_forecastable(step):
    return (0.0, GRAD_NOISE_MAX * _ramp(step), 0.0)

def instability_unforecastable(step):
    return (0.0, GRAD_NOISE_MAX * _instant(step), 0.0)

def label_noise_forecastable(step):
    return (0.0, 0.0, LABEL_FLIP_PROB * _ramp(step))

def label_noise_unforecastable(step):
    return (0.0, 0.0, LABEL_FLIP_PROB * _instant(step))

CONDITIONS = {
    "drift_forecastable": drift_forecastable, "drift_unforecastable": drift_unforecastable,
    "instability_forecastable": instability_forecastable, "instability_unforecastable": instability_unforecastable,
    "label_noise_forecastable": label_noise_forecastable, "label_noise_unforecastable": label_noise_unforecastable,
}
FRESH_SEEDS = list(range(100, 120))
print(f"{len(FRESH_SEEDS)} fresh seeds: {FRESH_SEEDS[0]}-{FRESH_SEEDS[-1]} (never used in any prior notebook this session)")


20 fresh seeds: 100-119 (never used in any prior notebook this session)


## 2. auto_grad, re-run on fresh seeds (R3-corrected calibration)

In [3]:
def auto_select_intervention_grad(loss_window, gradnorm_window, baseline_loss_mean, baseline_gradnorm_mean,
                                   sensitivity=GRAD_RATIO_SENSITIVITY):
    loss_ratio = (float(np.mean(loss_window)) + 1e-6) / (baseline_loss_mean + 1e-6)
    gradnorm_ratio = (float(np.mean(gradnorm_window)) + 1e-6) / (baseline_gradnorm_mean + 1e-6)
    return "reduce" if gradnorm_ratio > sensitivity * loss_ratio else "increase"

def run_trial_autograd(seed, disturbance_fn):
    rng = np.random.default_rng(seed)
    model = make_model(seed)
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.BCEWithLogitsLoss()
    loss_hist, gradnorm_hist = [], []
    threshold = baseline_loss_mean = baseline_gradnorm_mean = None
    for step in range(N_STEPS):
        drift_frac, grad_noise_extra, flip_prob = disturbance_fn(step)
        xb, yb = gen_batch(rng, drift_frac=drift_frac, flip_prob=flip_prob)
        opt.zero_grad()
        hidden = model[1](model[0](xb))
        logits = model[2](hidden).squeeze(-1)
        loss = loss_fn(logits, yb)
        loss.backward()
        loss_t = float(loss.item())
        if grad_noise_extra > 0:
            with torch.no_grad():
                for p in model.parameters():
                    if p.grad is not None:
                        p.grad.add_(torch.randn_like(p.grad) * grad_noise_extra)
        gn_t = grad_norm(model)
        loss_hist.append(loss_t)
        gradnorm_hist.append(gn_t)
        if step == CALIB_END - 1:
            calib_loss = np.array(loss_hist[CALIB_START:CALIB_END])
            threshold = float(calib_loss.mean() + 3 * calib_loss.std() + 1e-6)
            baseline_loss_mean = float(calib_loss.mean())
            baseline_gradnorm_mean = float(np.mean(gradnorm_hist[CALIB_START:CALIB_END]))
        if step >= CALIB_END and threshold is not None and loss_t > threshold:
            loss_window = loss_hist[max(0, step - TREND_WINDOW + 1):step + 1]
            gradnorm_window = gradnorm_hist[max(0, step - TREND_WINDOW + 1):step + 1]
            diagnosis = auto_select_intervention_grad(loss_window, gradnorm_window, baseline_loss_mean, baseline_gradnorm_mean)
            return diagnosis
        opt.step()
    return None

expected_grad = {"drift_forecastable": "increase", "drift_unforecastable": "increase",
                  "instability_forecastable": "reduce", "instability_unforecastable": "reduce"}

grad_results = []
for cond in ("drift_forecastable", "drift_unforecastable", "instability_forecastable", "instability_unforecastable"):
    fn = CONDITIONS[cond]
    for seed in FRESH_SEEDS:
        diagnosis = run_trial_autograd(seed, fn)
        grad_results.append({"condition": cond, "seed": seed, "diagnosis": diagnosis,
                              "correct": (diagnosis == expected_grad[cond]) if diagnosis else None})

grad_df = pd.DataFrame(grad_results)
grad_acc = grad_df["correct"].map({True: 1.0, False: 0.0}).mean()
n_untriggered = grad_df["diagnosis"].isna().sum()
print(f"auto_grad on {len(FRESH_SEEDS)} FRESH seeds (100-119): overall accuracy = {grad_acc*100:.1f}% "
      f"({n_untriggered} never triggered)")
for cond in expected_grad:
    sub = grad_df[grad_df["condition"] == cond]
    acc = sub["correct"].map({True: 1.0, False: 0.0}).mean()
    n_trig = sub["diagnosis"].notna().sum()
    print(f"  {cond:28s} n_triggered={n_trig}/{len(FRESH_SEEDS)}  accuracy={acc*100:.1f}%")


auto_grad on 20 FRESH seeds (100-119): overall accuracy = 98.8% (0 never triggered)
  drift_forecastable           n_triggered=20/20  accuracy=100.0%
  drift_unforecastable         n_triggered=20/20  accuracy=100.0%
  instability_forecastable     n_triggered=20/20  accuracy=95.0%
  instability_unforecastable   n_triggered=20/20  accuracy=100.0%


## 3. Val-gap diagnostic (EXP-008), re-run on fresh seeds (unchanged calibration/logic)

In [4]:
GAP_WINDOW = 10
VAL_BATCH_SIZE = 64
GAP_Z_THRESHOLD = 2.0

def auto_select_intervention_valgap(train_window, heldout_window, baseline_gap_mean, baseline_gap_std,
                                     z_threshold=GAP_Z_THRESHOLD):
    current_gap = float(np.mean(train_window)) - float(np.mean(heldout_window))
    z = (current_gap - baseline_gap_mean) / (baseline_gap_std + 1e-6)
    return "reduce" if z > z_threshold else "increase"

def run_trial_valgap(seed, disturbance_fn):
    train_rng = np.random.default_rng(seed)
    val_rng = np.random.default_rng(seed + 1_000_000)
    model = make_model(seed)
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.BCEWithLogitsLoss()
    train_loss_hist, heldout_loss_hist = [], []
    threshold = baseline_gap_mean = baseline_gap_std = None
    for step in range(N_STEPS):
        drift_frac, grad_noise_extra, flip_prob = disturbance_fn(step)
        xb, yb = gen_batch(train_rng, drift_frac=drift_frac, flip_prob=flip_prob)
        opt.zero_grad()
        hidden = model[1](model[0](xb))
        logits = model[2](hidden).squeeze(-1)
        loss = loss_fn(logits, yb)
        loss.backward()
        train_loss_t = float(loss.item())
        if grad_noise_extra > 0:
            with torch.no_grad():
                for p in model.parameters():
                    if p.grad is not None:
                        p.grad.add_(torch.randn_like(p.grad) * grad_noise_extra)
        with torch.no_grad():
            xv, yv = gen_batch(val_rng, drift_frac=drift_frac, flip_prob=0.0, batch_size=VAL_BATCH_SIZE)
            v_hidden = model[1](model[0](xv))
            v_logits = model[2](v_hidden).squeeze(-1)
            heldout_loss_t = float(loss_fn(v_logits, yv).item())
        train_loss_hist.append(train_loss_t)
        heldout_loss_hist.append(heldout_loss_t)
        if step == WARMUP:
            warm_train = np.array(train_loss_hist[:WARMUP])
            warm_gap = np.array(train_loss_hist[:WARMUP]) - np.array(heldout_loss_hist[:WARMUP])
            threshold = float(warm_train.mean() + 3 * warm_train.std() + 1e-6)
            baseline_gap_mean = float(warm_gap.mean())
            baseline_gap_std = float(warm_gap.std()) + 1e-6
        if step >= WARMUP and threshold is not None and train_loss_t > threshold:
            train_window = train_loss_hist[max(0, step - GAP_WINDOW + 1):step + 1]
            heldout_window = heldout_loss_hist[max(0, step - GAP_WINDOW + 1):step + 1]
            diag_gap = auto_select_intervention_valgap(train_window, heldout_window, baseline_gap_mean, baseline_gap_std)
            gap_now = float(np.mean(train_window)) - float(np.mean(heldout_window))
            return {"trigger_step": step, "diag_gap": diag_gap, "gap_now": gap_now}
        opt.step()
    return {"trigger_step": None, "diag_gap": None, "gap_now": None}

valgap_results = []
for cond_name, fn in CONDITIONS.items():
    for seed in FRESH_SEEDS:
        r = run_trial_valgap(seed, fn)
        valgap_results.append({"condition": cond_name, "seed": seed, "trigger_step": r["trigger_step"],
                                "diag_gap": r["diag_gap"], "gap_now": r["gap_now"]})

valgap_df = pd.DataFrame(valgap_results)
print(f"Total val-gap trials: {len(valgap_df)}")
print(f"{'condition':28s}{'n_trig':>8s}{'gap->reduce%':>14s}{'mean_gap_now':>14s}")
for cond in CONDITIONS:
    sub = valgap_df[(valgap_df["condition"] == cond) & valgap_df["trigger_step"].notna()]
    n = len(sub)
    gap_reduce_pct = (sub["diag_gap"] == "reduce").mean() * 100 if n else float("nan")
    mean_gap = sub["gap_now"].mean() if n else float("nan")
    print(f"{cond:28s}{n:8d}{gap_reduce_pct:14.0f}{mean_gap:14.4f}")


Total val-gap trials: 120
condition                     n_trig  gap->reduce%  mean_gap_now
drift_forecastable                20            15       -0.0082
drift_unforecastable              20            10        0.0093
instability_forecastable          20            30        0.0485
instability_unforecastable        20             5       -0.0276
label_noise_forecastable          20           100        0.2775
label_noise_unforecastable        20           100        0.1764


## 4. Interpretation: do the original 10-seed numbers hold on fresh seeds?

In [5]:
ORIGINAL_GRAD_ACC = 0.922  # R3-corrected overall accuracy from exp_auto_grad_sensitivity.ipynb, seeds 0-9
ORIGINAL_VALGAP_TP = {"label_noise_forecastable": 1.00, "label_noise_unforecastable": 1.00}
ORIGINAL_VALGAP_FP = {"drift_forecastable": 0.40, "instability_forecastable": 0.40,
                       "instability_unforecastable": 0.30, "drift_unforecastable": 0.00}

fresh_grad_acc = grad_acc

fresh_valgap = {}
for cond in CONDITIONS:
    sub = valgap_df[(valgap_df["condition"] == cond) & valgap_df["trigger_step"].notna()]
    fresh_valgap[cond] = (sub["diag_gap"] == "reduce").mean() if len(sub) else float("nan")

grad_diff = fresh_grad_acc - ORIGINAL_GRAD_ACC

interpretation = f'''
R6 -- DO THE ORIGINAL 10-SEED (0-9) NUMBERS HOLD ON 20 FRESH, NEVER-USED SEEDS (100-119)?

auto_grad overall accuracy: original (seeds 0-9, R3-corrected) = {ORIGINAL_GRAD_ACC*100:.1f}%,
fresh (seeds 100-119) = {fresh_grad_acc*100:.1f}% (difference = {grad_diff*100:+.1f} points).
{"HOLDS: within a few points, not a seed-selection artifact." if abs(grad_diff) < 0.10 else "DOES NOT CLEANLY HOLD: more than a 10-point swing on fresh seeds -- the original accuracy estimate may have been optimistic, or fresh seeds are hitting a different part of the noise distribution. Needs more seeds to resolve which."}

val-gap true-positive rate (label noise conditions), original vs fresh:
'''
for cond, orig in ORIGINAL_VALGAP_TP.items():
    interpretation += f"  {cond:28s} original={orig*100:.0f}%  fresh={fresh_valgap[cond]*100:.0f}%\n"

interpretation += "\nval-gap false-positive rate (non-target conditions), original vs fresh:\n"
for cond, orig in ORIGINAL_VALGAP_FP.items():
    interpretation += f"  {cond:28s} original={orig*100:.0f}%  fresh={fresh_valgap[cond]*100:.0f}%\n"

max_fp_swing = max(abs(fresh_valgap[c] - orig) for c, orig in ORIGINAL_VALGAP_FP.items())
interpretation += f"\nLargest false-positive-rate swing between original and fresh seeds: {max_fp_swing*100:.0f} points.\n"
interpretation += (
    "The val-gap diagnostic's sensitivity (100% TP on label noise) and its qualitative\n"
    "false-positive pattern (drift_unforecastable clean, the other three non-clean)\n"
    "generally " + ("HOLD" if max_fp_swing < 0.20 else "DO NOT CLEANLY HOLD") +
    " on fresh seeds -- " +
    ("this was not an artifact of the specific 10 seeds used during development." if max_fp_swing < 0.20 else
     "the false-positive rate is more seed-sensitive than the original 10-seed estimate suggested, which matters because the false-positive rate is precisely the property R1's correction already flagged as too high to deploy as-is.")
)
interpretation += (
    "\n\nCAVEAT: both diagnostics were tested on only 20 fresh seeds here -- this improves\n"
    "on the original 10 but is still a small-sample check, not a proper held-out test set\n"
    "with pre-registered acceptance criteria (that remains part of R13's broader ask for\n"
    "confidence intervals on every number carried into plan.doc, not yet done)."
)
print(interpretation)



R6 -- DO THE ORIGINAL 10-SEED (0-9) NUMBERS HOLD ON 20 FRESH, NEVER-USED SEEDS (100-119)?

auto_grad overall accuracy: original (seeds 0-9, R3-corrected) = 92.2%,
fresh (seeds 100-119) = 98.8% (difference = +6.6 points).
HOLDS: within a few points, not a seed-selection artifact.

val-gap true-positive rate (label noise conditions), original vs fresh:
  label_noise_forecastable     original=100%  fresh=100%
  label_noise_unforecastable   original=100%  fresh=100%

val-gap false-positive rate (non-target conditions), original vs fresh:
  drift_forecastable           original=40%  fresh=15%
  instability_forecastable     original=40%  fresh=30%
  instability_unforecastable   original=30%  fresh=5%
  drift_unforecastable         original=0%  fresh=10%

Largest false-positive-rate swing between original and fresh seeds: 25 points.
The val-gap diagnostic's sensitivity (100% TP on label noise) and its qualitative
false-positive pattern (drift_unforecastable clean, the other three non-clean)
